# HyPE Ablation Study Analysis

This notebook analyzes the results of the ablation study for the HyPE (Hypothetical Prompt Engineering) meta-prompt variants.

## Factors analyzed:
- **TF (Target Form)**: `inst` vs `hyp_inst`
- **R (Include Role)**: Whether to include the role section
- **US (Use Sections)**: Whether to use structured sections (TC, RS, OS)
- **TC (Task Context)**: Include task context section
- **RS (Role Section)**: Include role section in meta-prompt
- **OS (Output Section)**: Include output format section
- **MD (Markdown)**: Always 0 (disabled)

## Benchmarks:
- gsm8k (Exact Match)
- squad_v2 (BertScore)
- common_gen (BertScore)
- tweeteval (F1)
- xsum (BertScore)

In [ ]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.stats import hmean
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

In [ ]:
# Load results
results_path = Path("../ablation_prompts/ablation_scores.json")
with open(results_path) as f:
    data = json.load(f)

print(f"Loaded results from: {results_path}")
print(f"Meta info: {data['meta']}")
print(f"\nNumber of variants: {len(data['results'])}")

In [ ]:
# Parse variant names and extract factor values
def parse_variant_name(name: str) -> dict:
    """Parse variant name like 'TFhyp_inst_R0_US0_TC0_RS0_OS0_MD0' into factors."""
    parts = name.split('_')
    result = {}
    for part in parts:
        if part.startswith('TF'):
            result['TF'] = part.replace('TF', '')
        elif part.startswith('R'):
            result['R'] = int(part[1:])
        elif part.startswith('US'):
            result['US'] = int(part[2:])
        elif part.startswith('TC'):
            result['TC'] = int(part[2:])
        elif part.startswith('RS'):
            result['RS'] = int(part[2:])
        elif part.startswith('OS'):
            result['OS'] = int(part[2:])
        elif part.startswith('MD'):
            result['MD'] = int(part[2:])
    return result

# Build DataFrame from results
rows = []
for variant_name, variant_data in data['results'].items():
    factors = parse_variant_name(variant_name)
    
    for bench_name, bench_data in variant_data.get('benchmarks', {}).items():
        metric_value = bench_data.get('metric_value')
        format_compliance = bench_data.get('format_compliance', 0.0)
        
        # Skip failed entries
        if metric_value is None:
            continue
            
        row = {
            'variant': variant_name,
            'benchmark': bench_name,
            'metric_value': metric_value,
            'format_compliance': format_compliance,
            **factors
        }
        rows.append(row)

df = pd.DataFrame(rows)
print(f"Total rows (variant × benchmark): {len(df)}")
print(f"Unique variants: {df['variant'].nunique()}")
print(f"Unique benchmarks: {df['benchmark'].nunique()}")
df.head(10)

In [ ]:
# Check which variants have complete data
variant_counts = df.groupby('variant')['benchmark'].count()
print("Variants with complete benchmark coverage:")
complete_variants = variant_counts[variant_counts == 5].index.tolist()
print(f"  {len(complete_variants)} / {len(variant_counts)} variants have all 5 benchmarks")

print("\nVariants with missing benchmarks:")
incomplete = variant_counts[variant_counts < 5]
for var, count in incomplete.items():
    missing = set(df['benchmark'].unique()) - set(df[df['variant'] == var]['benchmark'])
    print(f"  {var}: {count}/5 benchmarks, missing: {missing}")

## 1. Per-Variant Metrics Table

In [ ]:
# Pivot table: variants × benchmarks with metric values
variant_bench_pivot = df.pivot_table(
    index='variant', 
    columns='benchmark', 
    values='metric_value',
    aggfunc='mean'
)

# Calculate average quality across benchmarks
variant_bench_pivot['avg_quality'] = variant_bench_pivot.mean(axis=1)

# Calculate harmonic mean of quality metrics
def calc_harmonic_mean(row):
    values = row.drop('avg_quality').values
    valid = values[~np.isnan(values)]
    if len(valid) == 0:
        return np.nan
    return hmean(valid)

variant_bench_pivot['harmonic_mean'] = variant_bench_pivot.apply(calc_harmonic_mean, axis=1)

# Add format compliance (average across benchmarks)
fmt_pivot = df.pivot_table(
    index='variant', 
    columns='benchmark', 
    values='format_compliance',
    aggfunc='mean'
)
variant_bench_pivot['avg_format_compliance'] = fmt_pivot.mean(axis=1)

# Final score: harmonic mean × average format compliance
variant_bench_pivot['final_score'] = variant_bench_pivot['harmonic_mean'] * variant_bench_pivot['avg_format_compliance']

# Sort by final score
variant_bench_pivot = variant_bench_pivot.sort_values('final_score', ascending=False)

print("Per-variant metrics (sorted by final_score):\n")
variant_bench_pivot.round(4)

In [ ]:
# Show top 10 variants
print("Top 10 variants by final_score:\n")
display_cols = ['avg_quality', 'harmonic_mean', 'avg_format_compliance', 'final_score']
variant_bench_pivot[display_cols].head(10).round(4)

## 2. Per-Factor Analysis

Analyze the effect of each factor (on/off) on the average metric value.

In [ ]:
# Per-factor analysis
factors = ['TF', 'R', 'US', 'TC', 'RS', 'OS']

factor_analysis = {}
for factor in factors:
    if factor == 'TF':
        # Special handling for TF (categorical)
        grouped = df.groupby('TF')['metric_value'].agg(['mean', 'std', 'count'])
    else:
        grouped = df.groupby(factor)['metric_value'].agg(['mean', 'std', 'count'])
    factor_analysis[factor] = grouped
    
print("## Factor Impact Analysis\n")
for factor, stats in factor_analysis.items():
    print(f"### {factor}")
    print(stats.round(4))
    print()

In [ ]:
# Visualize factor impact
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, factor in enumerate(factors):
    ax = axes[idx]
    
    if factor == 'TF':
        stats = factor_analysis[factor]
        bars = ax.bar(stats.index.astype(str), stats['mean'], yerr=stats['std'], capsize=5)
        ax.set_xlabel('Target Form')
    else:
        stats = factor_analysis[factor]
        bars = ax.bar(['Off (0)', 'On (1)'], stats['mean'], yerr=stats['std'], capsize=5)
        ax.set_xlabel(factor)
    
    ax.set_ylabel('Avg Metric Value')
    ax.set_title(f'Impact of {factor}')
    ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig('factor_impact.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved factor_impact.png")

## 3. Per-Benchmark Breakdown

In [ ]:
# Per-benchmark statistics
bench_stats = df.groupby('benchmark')['metric_value'].agg(['mean', 'std', 'min', 'max', 'count'])
bench_stats = bench_stats.sort_values('mean', ascending=False)
print("Benchmark-level statistics:\n")
bench_stats.round(4)

In [ ]:
# Visualize benchmark performance
fig, ax = plt.subplots(figsize=(10, 6))

bench_means = df.groupby('benchmark')['metric_value'].mean().sort_values(ascending=True)
bench_stds = df.groupby('benchmark')['metric_value'].std()

bars = ax.barh(bench_means.index, bench_means.values, xerr=bench_stds[bench_means.index], capsize=5)
ax.set_xlabel('Average Metric Value')
ax.set_title('Performance by Benchmark')
ax.set_xlim(0, 1)

for bar, mean_val in zip(bars, bench_means.values):
    ax.text(mean_val + 0.02, bar.get_y() + bar.get_height()/2, f'{mean_val:.3f}', va='center')

plt.tight_layout()
plt.savefig('benchmark_performance.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved benchmark_performance.png")

## 4. Format Compliance Analysis

In [ ]:
# Format compliance by benchmark
fmt_by_bench = df.groupby('benchmark')['format_compliance'].agg(['mean', 'std', 'min', 'max'])
print("Format Compliance by Benchmark:\n")
fmt_by_bench.round(4)

In [ ]:
# Format compliance by variant
fmt_by_variant = df.groupby('variant')['format_compliance'].mean().sort_values(ascending=False)
print("Format Compliance by Variant (top 10):\n")
fmt_by_variant.head(10).round(4)

In [ ]:
# Visualize format compliance
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# By benchmark
ax1 = axes[0]
fmt_bench = df.groupby('benchmark')['format_compliance'].mean().sort_values()
ax1.barh(fmt_bench.index, fmt_bench.values)
ax1.set_xlabel('Format Compliance')
ax1.set_title('Format Compliance by Benchmark')
ax1.set_xlim(0, 1.1)

# By variant (top 15)
ax2 = axes[1]
fmt_var = df.groupby('variant')['format_compliance'].mean().sort_values(ascending=False).head(15)
ax2.barh(range(len(fmt_var)), fmt_var.values)
ax2.set_yticks(range(len(fmt_var)))
ax2.set_yticklabels([v[:30] + '...' if len(v) > 30 else v for v in fmt_var.index], fontsize=8)
ax2.set_xlabel('Format Compliance')
ax2.set_title('Format Compliance by Variant (Top 15)')
ax2.set_xlim(0, 1.1)

plt.tight_layout()
plt.savefig('format_compliance.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved format_compliance.png")

## 5. Factor Interaction Analysis

Analyze how combinations of factors affect performance.

In [ ]:
# US (Use Sections) impact - this is the main toggle
us_impact = df.groupby('US')['metric_value'].agg(['mean', 'std', 'count'])
print("US (Use Sections) Impact:\n")
print(us_impact.round(4))

# When US=1, breakdown by TC, RS, OS
print("\n--- When US=1 (sections enabled) ---\n")
df_us1 = df[df['US'] == 1]

for factor in ['TC', 'RS', 'OS']:
    stats = df_us1.groupby(factor)['metric_value'].agg(['mean', 'std', 'count'])
    print(f"{factor}:")
    print(stats.round(4))
    print()

In [ ]:
# Interaction heatmap: US × R
pivot_us_r = df.pivot_table(index='US', columns='R', values='metric_value', aggfunc='mean')
print("US × R Interaction:\n")
print(pivot_us_r.round(4))

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(pivot_us_r, annot=True, fmt='.3f', cmap='viridis', ax=ax)
ax.set_title('Metric Value: US × R Interaction')
ax.set_xlabel('R (Include Role)')
ax.set_ylabel('US (Use Sections)')
plt.tight_layout()
plt.savefig('interaction_us_r.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Full factor correlation matrix
factor_cols = ['R', 'US', 'TC', 'RS', 'OS']
corr_with_metric = {}
for col in factor_cols:
    corr_with_metric[col] = df[col].corr(df['metric_value'])

corr_df = pd.DataFrame.from_dict(corr_with_metric, orient='index', columns=['correlation_with_metric'])
corr_df = corr_df.sort_values('correlation_with_metric', key=abs, ascending=False)
print("Factor correlation with metric value:\n")
corr_df.round(4)

## 6. Final Ranking with Harmonic Mean

In [ ]:
# Final ranking table
ranking = variant_bench_pivot[['harmonic_mean', 'avg_format_compliance', 'final_score']].copy()
ranking = ranking.sort_values('final_score', ascending=False)
ranking['rank'] = range(1, len(ranking) + 1)
ranking = ranking[['rank', 'harmonic_mean', 'avg_format_compliance', 'final_score']]

print("Final Ranking (by Final Score = Harmonic Mean × Format Compliance):\n")
ranking.head(20).round(4)

In [ ]:
# Visualize top 15 variants
fig, ax = plt.subplots(figsize=(12, 8))

top15 = ranking.head(15)
y_pos = range(len(top15))

bars = ax.barh(y_pos, top15['final_score'].values)
ax.set_yticks(y_pos)
ax.set_yticklabels([f"#{i+1} {idx[:35]}..." if len(idx) > 35 else f"#{i+1} {idx}" for i, idx in enumerate(top15.index)], fontsize=8)
ax.invert_yaxis()
ax.set_xlabel('Final Score')
ax.set_title('Top 15 Variants by Final Score\n(Harmonic Mean × Format Compliance)')

for i, (bar, val) in enumerate(zip(bars, top15['final_score'].values)):
    ax.text(val + 0.005, i, f'{val:.4f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('top_variants.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved top_variants.png")

## 7. Summary Statistics

In [ ]:
# Summary statistics
print("=" * 60)
print("ABLATION STUDY SUMMARY")
print("=" * 60)

print(f"\nTotal variants evaluated: {df['variant'].nunique()}")
print(f"Total benchmark evaluations: {len(df)}")
print(f"Benchmarks: {df['benchmark'].unique().tolist()}")

print(f"\n--- Quality Metrics ---")
print(f"Average metric value: {df['metric_value'].mean():.4f}")
print(f"Std deviation: {df['metric_value'].std():.4f}")
print(f"Min: {df['metric_value'].min():.4f}")
print(f"Max: {df['metric_value'].max():.4f}")

print(f"\n--- Format Compliance ---")
print(f"Average format compliance: {df['format_compliance'].mean():.4f}")
print(f"Min: {df['format_compliance'].min():.4f}")
print(f"Max: {df['format_compliance'].max():.4f}")

print(f"\n--- Best Variant ---")
best_variant = ranking.index[0]
print(f"Variant: {best_variant}")
print(f"Final Score: {ranking.iloc[0]['final_score']:.4f}")
print(f"Harmonic Mean: {ranking.iloc[0]['harmonic_mean']:.4f}")
print(f"Format Compliance: {ranking.iloc[0]['avg_format_compliance']:.4f}")

print(f"\n--- Factor Impact Summary ---")
for factor in ['R', 'US', 'TC', 'RS', 'OS']:
    on_val = df[df[factor] == 1]['metric_value'].mean()
    off_val = df[df[factor] == 0]['metric_value'].mean()
    diff = on_val - off_val
    direction = "↑" if diff > 0 else "↓"
    print(f"{factor}: On={on_val:.4f}, Off={off_val:.4f}, Diff={diff:+.4f} {direction}")

print("\n" + "=" * 60)

In [ ]:
# Export summary to CSV
ranking.to_csv('ablation_ranking.csv')
print("Exported ranking to ablation_ranking.csv")

# Export per-benchmark breakdown
variant_bench_pivot.to_csv('ablation_variant_benchmarks.csv')
print("Exported variant × benchmark metrics to ablation_variant_benchmarks.csv")